# Modele préféré des archivistes

Comme il s'agit d'une évaluation faite par plusieurs archivistes (les memes) sur plusieurs textes (les memes) et qu'il peut y avoir des trous dans les données, on doit utiliser un modele qui est un **mixed effect model** pour savoir quel est le modele effectivement préféré par les archivistes.

In [33]:
import numpy as np
import pandas as pd
import statsmodels as sm
import statsmodels.formula.api as smf
import sqlite3

In [7]:
## Récupère les vraies données depuis la db
db = "./model_ranking/backend/databases/unified.db"
with sqlite3.connect(db) as conn:
    ranking = pd.read_sql(
        """
        SELECT r.id, m.alias, r.usrname, r.ranking
        FROM ranking r JOIN models m ON r.model = m.name""",
        conn)
ranking

,id,alias,usrname,ranking


In [85]:
## Comme on n'a pas de vraies donnéees (du moins pas ici), je vais en simuler
usrnames   = pd.Series([ "rolande", "klaas", "gert", "annelies", "raphaelle" ])
with sqlite3.connect(db) as conn:
    modelnames = pd.read_sql("select alias from models", conn)

## on compte 300 data points
nb_points = 300
#
docs     = np.random.randint(0, high=420, size=(nb_points,))
models   = modelnames['alias'].iloc[np.random.randint(0, high=len(modelnames), size=(nb_points,))]
usrs     = usrnames.iloc[np.random.randint(0, high=len(usrnames), size=(nb_points,))]
rankings = np.random.randint(0, high=6, size=(nb_points,))

ranking  = pd.DataFrame(np.stack([docs, models, usrs, rankings], axis=1), columns=["id", "alias", "usrname", "ranking"])
ranking

,id,alias,usrname,ranking
0,276,llama,rolande,1
1,178,llama,rolande,4
2,233,mistralnemo,raphaelle,1
3,307,mistral7b,annelies,5
4,134,mistral7b,gert,0
...,...,...,...,...
295,137,llama,klaas,4
296,60,gpt4o-mini,raphaelle,5
297,79,vieille_france,raphaelle,1
298,182,mistralnemo,rolande,5


In [86]:
## conversion des do catégoriques en attr numériques
ranking["usrname"] = ranking["usrname"].astype("category")
ranking["alias"]   = ranking["alias"].astype("category")
ranking["id"]      = ranking["id"].astype("category")
ranking["ranking"] = ranking["ranking"].astype(np.float32)
ranking = ranking.rename(columns={"id": "document"})
ranking.dtypes

document    category
alias       category
usrname     category
ranking      float32
dtype: object

In [90]:
## on appplique le modele pour de vrai
# 1. La formule principale ne contient que les effets fixes
formula = "ranking ~ alias"

# 2. On définit les effets aléatoires dans un dictionnaire
# C'est la manière explicite de demander des intercepts aléatoires
vc_formula = {
    "document": "0 + C(document)",
    "usrname": "0 + C(usrname)"
}

model = smf.mixedlm(
    formula=formula,
    vc_formula=vc_formula,
    data=ranking,
    groups=ranking["document"])
results = model.fit()

## Comment comprendre le résultat

Le résulat de notre summary, il faut regarder la valeur de nos intercepts. 
Le modele se base par rapport à un model de reference (qui n'est pas affiché). 
Les lignes suivantes, donnent la différence par rapport à l'intercept de la référence. 
Si c'est positif, ca veut dire que l'effet qu'on cherche à prédire est meilleur que la réf.
Si c'est négatif, ca veut dire que l'effet qu'on cherche à montrer est moins bon que la réf;

**ATTENTION**
Pour pouvoir dire qu'un modele est vraiment meilleur que l'autre, on doit aussi regarder la valeur dans la colonne `P>|z|`.
Si cette valeur est > 0.05 ca veut dire que la différence n'est pas statistiquement significative.
Si cette valeur est < 0.05 ca veut dire qu'il y a vraiment un truc.

In [93]:
results.summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
                Mixed Linear Model Regression Results
=====================================================================
Model:                  MixedLM     Dependent Variable:     ranking  
No. Observations:       300         Method:                 REML     
No. Groups:             225         Scale:                  1.8956   
Min. group size:        1           Log-Likelihood:         -576.4360
Max. group size:        5           Converged:              Yes      
Mean group size:        1.3                                          
---------------------------------------------------------------------
                           Coef.  Std.Err.   z    P>|z| [0.025 0.975]
---------------------------------------------------------------------
Intercept                   2.578    0.261  9.888 0.000  2.067  3.089
alias[T.gpt4o-mini]        -0.166    0.364 -0.456 0.649 -0.878  0.547
alias[T.llama]             -0.073    0.360 -0.203 0.839 -0.780  0.633
alias[T.mistral7b]         -0.024    0.362 -0.067 0.947 -0.733  0.685
alias[T.mistralnemo]        0.071    0.353  0.201 0.841 -0.621  0.763
alias[T.vieille_france]    -0.328    0.353 -0.929 0.353 -1.020  0.364
alias[T.vieille_france_v2]  0.092    0.380  0.241 0.809 -0.654  0.837
document Var                0.525    0.285                           
usrname Var                 0.361    0.548                           
=====================================================================

"""

**Conclusion:** Ici aucun modele n'est meilleur que les autres (et c'est normal car on a tiré des valeurs au hasard dans une distrib uniforme).